In [ ]:
#pip install pandas==2.3.3

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 4.4 MB/s  0:00:02a 0:00:01m eta 0:00:01
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.6
    Uninstalling pandas-3.0.6:
      Successfully uninstalled pandas-3.0.6

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.types import  *
from pyspark.sql.functions import * 

In [11]:
spark = SparkSession.builder.master("local").config("spark.jars.packages" , "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0").getOrCreate()
#Spark varsayılan olarak Kafka sürücülerini içermediğinden, hem batch hem de streaming işlemlerinde Kafka ile iletişim kurabilmesi için gerekli bağlantı kütüphanesini oturuma dahil ettik.
#2.13: Spark 4.2.0'ın kullandığı Scala sürümü.

#4.2.0: Senin ortamındaki Spark sürümü.

#0-10: Paketin adındaki bu sabit ifade Kafka 0.10 ve üzeri tüm broker sürümleriyle (Kafka 4.4 dahil) uyumludur; değiştirilmez.

In [ ]:
#batch işlem yapılıyor yani şuana kadar olan verileri alıyoruz yeni eklenen veriler gelmez onun için de streaming yapmalıyız = readStream
df=spark.read.format("kafka").option("kafka.bootstrap.servers" , bootstrap_servers).option("subscribe" , "tweetModel").option("startingOffsets","earliest").load()

In [13]:
df.show() # göderdiğimiz verilerin bilgileri gelir value kısmı byte şeklindedir çünkü öyle göderilir

+----+--------------------+----------+---------+------+--------------------+-------------+
| key|               value|     topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------+---------+------+--------------------+-------------+
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     0|2026-11-16 14:23:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     1|2026-11-16 14:23:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     2|2026-11-16 14:23:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     3|2026-11-16 14:23:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     4|2026-11-16 14:23:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     5|2026-11-16 14:24:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     6|2026-11-16 14:24:...|            0|
|NULL|[7B 22 75 73 65 7...|tweetModel|        1|     7|2026-11-16 14:24:...|            0|

In [14]:
df = df.selectExpr("cast(value as string)" ) # sparkta action olunca verileri üzerinde yaptığımız değşiklik kaydedilir di bu yüzden kendisine eşitleyip actiona zorluyoruz

In [16]:
df.show(truncate=False) # tabloyu kırpmaması için

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                              

In [18]:
schema  =StructType([
    StructField("user_name" , StringType()),
    StructField("user_location" , StringType()),
    StructField("user_description" , StringType()),
    StructField("user_followers" , IntegerType()),
    StructField("user_friends" , IntegerType()),
    StructField("date" , DateType()),
    StructField("text" , StringType())
])

In [20]:
rawDF=df.withColumn("value2" , from_json(df["value"] , schema)).select("value2.*") # from_json fonku ile value columndaki dataları schema göre şekillendirip value2 col u olarak kaydet
#sonra select ile value2 deki tüm veirleri al ve onları rawDF e eşleştir

In [21]:
rawDF.show()

+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|           user_name|       user_location|    user_description|user_followers|user_friends|      date|                text|
+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|Best uncle on pla...|               "NaN"|               "NaN"|           741|         730|2021-10-06|That marble episo...|
|BarBiE Fábregas 🇿🇦|        South Africa|Legal Administrat...|          1877|        2057|2021-10-06|Please vote in my...|
|        Mega Visions|               "NaN"|The world's first...|         16613|         484|2021-10-06|That's definitely...|
|Samaanslife (⧖) ?...|K.HulhuMalé,Maldi...|19 | He/Him | 📸 ...|           825|        2477|2021-10-06|Dalgona Candy Lov...|
|  dredriven83➡️#NYCC|        Brooklyn, NY|I'm considered a ...|           416|        3028|2021-10-06|What wait there's...|
|  

In [22]:
rawDF.printSchema()

root
 |-- user_name: string (nullable = true)
 |-- user_location: string (nullable = true)
 |-- user_description: string (nullable = true)
 |-- user_followers: integer (nullable = true)
 |-- user_friends: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- text: string (nullable = true)



In [ ]:
duplicateDf=rawDF.dropDuplicates() # tekrarlanan verileri çıkarıp ham df i return eder

In [24]:
duplicateDf.show()

+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|           user_name|       user_location|    user_description|user_followers|user_friends|      date|                text|
+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|🏴󠁧󠁢󠁳󠁣󠁴󠁿han...|               "NaN"|mii2 , maxtul, dm...|            37|         207|2021-10-06|LMFAOOOOOOOOOOOO ...|
|Best uncle on pla...|               "NaN"|               "NaN"|           741|         730|2021-10-06|That marble episo...|
|             Charlie|      United Kingdom|Aspiring Youtuber...|            24|          47|2021-10-06|I Didn't mean to ...|
|       Paras Guglani|      Gurgaon, India|Just another geek...|          1080|          69|2021-10-06|Just spotted an a...|
|         My Big Bets|               "NaN"|Over 25 years of ...|         11879|        5272|2021-10-06|🏇RACING ABANDONE...|
|🎃EEEEEK

In [25]:
duplicateDf.count()

607

In [ ]:
rawDF.count() # yukarda duplicated olan veirlerin çıarılmış hali ile raw df count ları aynı demek ki duplicated veri yokmuş

607

In [28]:
rawDF.filter("user_followers > 10000").show()

+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|           user_name|       user_location|    user_description|user_followers|user_friends|      date|                text|
+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|        Mega Visions|               "NaN"|The world's first...|         16613|         484|2021-10-06|That's definitely...|
|        David Kappel|        South Africa|@iDiskiTimes | fo...|         14837|        2873|2021-10-06|So who has watch ...|
|         RVCJ Movies|              Mumbai|Official MOVIES A...|         28974|         226|2021-10-06|Squid Game!\n#squ...|
|  Tara Sharma Saluja|       Mumbai, India|Actress, Fool, Cr...|         44501|        1319|2021-10-06|Walking into #Wed...|
|         My Big Bets|               "NaN"|Over 25 years of ...|         11879|        5272|2021-10-06|🏇RACING ABANDONE...|
|

In [30]:
rawDF.filter("user_followers < 10").show()

+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|           user_name|       user_location|    user_description|user_followers|user_friends|      date|                text|
+--------------------+--------------------+--------------------+--------------+------------+----------+--------------------+
|        JustSomeGoth|England, United K...|23|Furry|Nonbinar...|             8|          52|2021-10-06|SQUIDGAME CHIBI Y...|
|          carib_gal1|               "NaN"|I'm aspired to be...|             0|           9|2021-10-06|Who saw squid gam...|
|          papi pique|               "NaN"|       pique sexy 🥵|             2|           8|2021-10-06|I mean squid game...|
|          ¯\_(ツ)_/¯|               "NaN"|                Hehe|             0|           0|2021-10-06|just thinking abo...|
|           ToDisenyo|               "NaN"|Design, Marketing...|             0|          14|2021-10-06|"Squid game" has ...|
| 

In [31]:
rawDF.groupBy("user_name").count().show() # user name göre tweetleri grupladık böylece biri birkaç defa konu hakkkında tweeet atmışmmı görebiliriz

+--------------------+-----+
|           user_name|count|
+--------------------+-----+
|       Digital Disco|    1|
|         Mr.Saxobeat|    1|
|     Rayhan Trimulya|    1|
|          papi pique|    1|
|   SILCREEK REALTORS|    1|
|  MetaMemeGuardian69|    1|
|      Second Opinion|    1|
|     ❂ Yübbîę Umoh ❂|    4|
|•Pull Up Game Crazy•|    1|
|                 nuc|    1|
|BarBiE Fábregas 🇿🇦|    1|
|               Chris|    1|
|                 mel|    1|
|             Jusreal|    1|
|              Turnip|    1|
|    Paperized Crafts|    1|
|      Sassy Infotech|    1|
|     Paul Woodthorpe|    1|
|           GURNOOR S|    1|
| Theofficial_kemjika|    1|
+--------------------+-----+
only showing top 20 rows
